In [1]:
import sys as __harness_sys
from pathlib import Path as __harness_Path
__harness_sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\tutorials')
__harness_sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\scripts')
from _offline import enforce_offline as __guard, blocked_attempts as __blocked
__guard()
import capture_runresults_213 as __capture
__capture.install()
import agentic_systems as __harness_toolkit
assert __harness_toolkit.__version__ == "2.1.3"
assert "site-packages" in __harness_Path(__harness_toolkit.__file__).parts

# Challenge completo: pedidos

Tools → Skills → varios Agents → System con decisiones; Environment y Eval sobre la misma declaración.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [2]:
import agentic_systems as toolkit


In [3]:
@toolkit.tool
def quote(units: int, unit_price: int) -> dict:
    if units <= 0 or unit_price < 0:
        raise ValueError("Cantidad o precio inválidos")
    return {"total": units * unit_price}

@toolkit.tool
def approve(total: int) -> dict:
    return {"total": total, "status": "approved"}

@toolkit.tool
def review(total: int) -> dict:
    return {"total": total, "status": "manual_review"}

pricing = toolkit.skill(name="pricing", tools=(quote,))
decisions = toolkit.skill(name="decisions", tools=(approve, review))

def route_order(data):
    return "approve" if data["total"] <= 1000 else "review"

records = ({"units": 2, "unit_price": 100}, {"units": 3, "unit_price": 500})
expected = [{"total": 200, "status": "approved"}, {"total": 1500, "status": "manual_review"}]


def build_orders(backend="native", *, runtime=None, framework=None):
    quoter = toolkit.Agent("quoter", toolkit.Action("quote", "quote"), skills=(pricing,))
    approver = toolkit.Agent("approver", toolkit.Action("approve", "approve"), skills=(decisions,))
    reviewer = toolkit.Agent("reviewer", toolkit.Action("review", "review"), skills=(decisions,))
    if runtime is not None:
        approver = runtime_step("approver", approve, runtime, framework)
        reviewer = runtime_step("reviewer", review, runtime, framework)
    return toolkit.System("orders", toolkit.Graph(
        nodes=(("quote", quoter), ("approve", approver), ("review", reviewer)),
        edges=(("quote", route_order), ("approve", "END"), ("review", "END")), start="quote",
    ), backend=backend)

def build_day(backend="native", *, runtime=None, framework=None):
    return toolkit.Environment("day", build_orders(backend, runtime=runtime, framework=framework),
        records=records, memory={"approved": 0},
        update=lambda memory, record, result: {
            "approved": memory["approved"] + int(result.data["status"] == "approved")
        })

def check_day(result, case):
    valid = result.data["outputs"] == expected and result.data["summary"]["memory"] == {"approved": 1}
    return {"ok": valid, "issues": [] if valid else ["Resultados o memoria incorrectos"],
            "metrics": {"approved": result.data["summary"]["memory"]["approved"]}}

def evaluate_day(unit):
    return toolkit.Eval("orders_acceptance", unit,
        cases=({"input": {}, "expected": {"ok": True}},), checks=(check_day,))



for backend in ("native", "langgraph"):
    day = build_day(backend)
    result = day.run()
    evaluation = evaluate_day(day).run()
    assert result.ok and evaluation.ok and check_day(result, {})["ok"]
    print({"backend": backend, "input": list(records), "process": "quote → decision → approve/review → memory → Eval", "output": result.data})

{'backend': 'native', 'input': [{'units': 2, 'unit_price': 100}, {'units': 3, 'unit_price': 500}], 'process': 'quote → decision → approve/review → memory → Eval', 'output': {'outputs': [{'total': 200, 'status': 'approved'}, {'total': 1500, 'status': 'manual_review'}], 'history': [{'step': 0, 'record': {'units': 2, 'unit_price': 100}, 'memory_before': {'approved': 0}, 'memory_after': {'approved': 1}, 'graph_state': {'run': {'text': '', 'final': {'total': 200, 'status': 'approved'}, 'data': {'total': 200, 'status': 'approved'}, 'ok': True, 'messages': [], 'raw_responses': [], 'usage': {'requests': 2, 'inference_requests': 0, 'tool_calls': 2}, 'engine': 'python-runtime', 'model': '', 'mode': 'default', 'trace_schema_version': 'agentic_systems.trace.v1', 'validation': None, 'errors': [], 'meta': {'name': 'orders', 'kind': 'system', 'backend': 'native', 'skills': []}, 'execution_id': 'run-682ac1ef06a349a297bac210586c3d1d', 'parent_execution_id': None, 'children': [{'text': '', 'final': {'to

{'backend': 'langgraph', 'input': [{'units': 2, 'unit_price': 100}, {'units': 3, 'unit_price': 500}], 'process': 'quote → decision → approve/review → memory → Eval', 'output': {'outputs': [{'total': 200, 'status': 'approved'}, {'total': 1500, 'status': 'manual_review'}], 'history': [{'step': 0, 'record': {'units': 2, 'unit_price': 100}, 'memory_before': {'approved': 0}, 'memory_after': {'approved': 1}, 'graph_state': {'run': {'text': '', 'final': {'total': 200, 'status': 'approved'}, 'data': {'total': 200, 'status': 'approved'}, 'ok': True, 'messages': [], 'raw_responses': [], 'usage': {'requests': 2, 'inference_requests': 0, 'tool_calls': 2}, 'engine': 'python-runtime', 'model': '', 'mode': 'default', 'trace_schema_version': 'agentic_systems.trace.v1', 'validation': None, 'errors': [], 'meta': {'name': 'orders', 'kind': 'system', 'backend': 'langgraph', 'skills': []}, 'execution_id': 'run-dc8afeb9c1c849ccbf59b04fe05324c8', 'parent_execution_id': None, 'children': [{'text': '', 'final'

In [4]:
__capture.save('C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\revisions\\2.1.3\\learning-evidence-elevated\\notebook-runresults\\20261023T044738543256\\11-07_order_challenge.json', 'tutorials/core/07_order_challenge.ipynb')
assert __blocked() == 0